# YOLOv8 Model Training on Google Colab

This notebook helps you download the public **SKU-110K** dataset and train a YOLOv8 model using Google Colab's GPU. After training, it packages the checkpoints (`best.pt`, `last.pt`) and evaluation plots into a ZIP archive for you to download back to your local setup.

### Important: Select a GPU Runtime
Before running the cells, make sure you are using a GPU runtime:
1. Go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (or any other available GPU).
3. Click **Save**.

In [ ]:
# Install the Ultralytics library
!pip install ultralytics

# Verify GPU access and library installation
import torch
from ultralytics import utils
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
utils.checks()

## 1. Train on SKU-110K Dataset

The SKU-110K dataset is a standard benchmark for detecting densely packed objects on retail shelves. Ultralytics has built-in support for auto-downloading and converting this dataset when using `data='sku110k.yaml'`.

*Note: The full dataset is about 11 GB, and downloading and training for 10 epochs on a T4 GPU will take approximately 45–60 minutes. If you want a quick 5-minute plumbing test instead, you can change the configuration to use `data='coco8.yaml'` or `data='coco128.yaml'`.*

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 nano detection model
model = YOLO('yolov8n.pt')

# Start training on SKU-110K
# Change data='sku110k.yaml' to data='coco128.yaml' if you want a faster demo run.
results = model.train(
    data='sku110k.yaml',
    epochs=10,
    imgsz=640,
    batch=16,
    device=0,
    project='sku110k_runs',
    name='retail_finetune'
)

## 2. Compress and Download Results

Once training completes, this cell will package all model checkpoints (including `best.pt` and `last.pt`), results CSV, and validation plots into a single ZIP file and trigger a download.

In [ ]:
import zipfile
import os
from google.colab import files

def zip_folder(folder_path, output_path):
    with zipfile.ZipFile(output_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, filenames in os.walk(folder_path):
            for filename in filenames:
                full_path = os.path.join(root, filename)
                arcname = os.path.relpath(full_path, os.path.join(folder_path, '..'))
                zipf.write(full_path, arcname)

# Zip the runs directory containing weights and plots
zip_folder('sku110k_runs', 'sku110k_runs.zip')

# Trigger browser download of the zip file
files.download('sku110k_runs.zip')